# Using Jupyter Notebooks
:label:`sec_jupyter`


This section describes how to edit and run the code
in each section of this book
using the Jupyter Notebook. Make sure you have
installed Jupyter and downloaded the
code as described in
:ref:`chap_installation`.
If you want to know more about Jupyter see the excellent tutorial in
their [documentation](https://jupyter.readthedocs.io/en/latest/).


## Editing and Running the Code Locally

Suppose that the local path of the book's code is `xx/yy/d2l-en/`. Use the shell to change the directory to this path (`cd xx/yy/d2l-en`) and run the command `jupyter notebook`. If your browser does not do this automatically, open http://localhost:8888 and you will see the interface of Jupyter and all the folders containing the code of the book, as shown in :numref:`fig_jupyter00`.

![The folders containing the code of this book.](https://github.com/d2l-ai/d2l-en-colab/blob/master/img/jupyter00.png?raw=1)
:width:`600px`
:label:`fig_jupyter00`


You can access the notebook files by clicking on the folder displayed on the webpage.
They usually have the suffix ".ipynb".
For the sake of brevity, we create a temporary "test.ipynb" file.
The content displayed after you click it is
shown in :numref:`fig_jupyter01`.
This notebook includes a markdown cell and a code cell. The content in the markdown cell includes "This Is a Title" and "This is text.".
The code cell contains two lines of Python code.

![Markdown and code cells in the "text.ipynb" file.](https://github.com/d2l-ai/d2l-en-colab/blob/master/img/jupyter01.png?raw=1)
:width:`600px`
:label:`fig_jupyter01`


Double click on the markdown cell to enter edit mode.
Add a new text string "Hello world." at the end of the cell, as shown in :numref:`fig_jupyter02`.

![Edit the markdown cell.](https://github.com/d2l-ai/d2l-en-colab/blob/master/img/jupyter02.png?raw=1)
:width:`600px`
:label:`fig_jupyter02`


As demonstrated in :numref:`fig_jupyter03`,
click "Cell" $\rightarrow$ "Run Cells" in the menu bar to run the edited cell.

![Run the cell.](https://github.com/d2l-ai/d2l-en-colab/blob/master/img/jupyter03.png?raw=1)
:width:`600px`
:label:`fig_jupyter03`

After running, the markdown cell is shown in :numref:`fig_jupyter04`.

![The markdown cell after running.](https://github.com/d2l-ai/d2l-en-colab/blob/master/img/jupyter04.png?raw=1)
:width:`600px`
:label:`fig_jupyter04`


Next, click on the code cell. Multiply the elements by 2 after the last line of code, as shown in :numref:`fig_jupyter05`.

![Edit the code cell.](https://github.com/d2l-ai/d2l-en-colab/blob/master/img/jupyter05.png?raw=1)
:width:`600px`
:label:`fig_jupyter05`


You can also run the cell with a shortcut ("Ctrl + Enter" by default) and obtain the output result from :numref:`fig_jupyter06`.

![Run the code cell to obtain the output.](https://github.com/d2l-ai/d2l-en-colab/blob/master/img/jupyter06.png?raw=1)
:width:`600px`
:label:`fig_jupyter06`


When a notebook contains more cells, we can click "Kernel" $\rightarrow$ "Restart & Run All" in the menu bar to run all the cells in the entire notebook. By clicking "Help" $\rightarrow$ "Edit Keyboard Shortcuts" in the menu bar, you can edit the shortcuts according to your preferences.

## Advanced Options

Beyond local editing two things are quite important: editing the notebooks in the markdown format and running Jupyter remotely.
The latter matters when we want to run the code on a faster server.
The former matters since Jupyter's native ipynb format stores a lot of auxiliary data that is
irrelevant to the content,
mostly related to how and where the code is run.
This is confusing for Git, making
reviewing contributions very difficult.
Fortunately there is an alternative---native editing in the markdown format.

### Markdown Files in Jupyter

If you wish to contribute to the content of this book, you need to modify the
source file (md file, not ipynb file) on GitHub.
Using the notedown plugin we
can modify notebooks in the md format directly in Jupyter.


First, install the notedown plugin, run the Jupyter Notebook, and load the plugin:

```
pip install d2l-notedown  # You may need to uninstall the original notedown.
jupyter notebook --NotebookApp.contents_manager_class='notedown.NotedownContentsManager'
```

You may also turn on the notedown plugin by default whenever you run the Jupyter Notebook.
First, generate a Jupyter Notebook configuration file (if it has already been generated, you can skip this step).

```
jupyter notebook --generate-config
```

Then, add the following line to the end of the Jupyter Notebook configuration file (for Linux or macOS, usually in the path `~/.jupyter/jupyter_notebook_config.py`):

```
c.NotebookApp.contents_manager_class = 'notedown.NotedownContentsManager'
```

After that, you only need to run the `jupyter notebook` command to turn on the notedown plugin by default.

### Running Jupyter Notebooks on a Remote Server

Sometimes, you may want to run Jupyter notebooks on a remote server and access it through a browser on your local computer. If Linux or macOS is installed on your local machine (Windows can also support this function through third-party software such as PuTTY), you can use port forwarding:

```
ssh myserver -L 8888:localhost:8888
```

The above string `myserver` is the address of the remote server.
Then we can use http://localhost:8888 to access the remote server `myserver` that runs Jupyter notebooks. We will detail on how to run Jupyter notebooks on AWS instances
later in this appendix.

### Timing

We can use the `ExecuteTime` plugin to time the execution of each code cell in Jupyter notebooks.
Use the following commands to install the plugin:

```
pip install jupyter_contrib_nbextensions
jupyter contrib nbextension install --user
jupyter nbextension enable execute_time/ExecuteTime
```

## Summary

* Using the Jupyter Notebook tool, we can edit, run, and contribute to each section of the book.
* We can run Jupyter notebooks on remote servers using port forwarding.


## Exercises

1. Edit and run the code in this book with the Jupyter Notebook on your local machine.
1. Edit and run the code in this book with the Jupyter Notebook *remotely* via port forwarding.
1. Compare the running time of the operations $\mathbf{A}^\top \mathbf{B}$ and $\mathbf{A} \mathbf{B}$ for two square matrices in $\mathbb{R}^{1024 \times 1024}$. Which one is faster?


[Discussions](https://discuss.d2l.ai/t/421)


In [15]:
import torch
import torch.nn as nn
import numpy as np
from torch.utils.data import Dataset, DataLoader
import copy
import os

# ==========================================
# 1. Load Data  (tx labels NOT used for training)
# ==========================================
rx_full = np.loadtxt('rx_noisy_pam2.dat') - 2.56

# tx kept ONLY for optional post-hoc BER evaluation — never seen during training
tx_raw  = np.loadtxt('tx_labels_pam2.dat')
tx_full = np.where(tx_raw == 1.0, 1.0, -1.0)[::2]

SPLIT_SAMPLES = int(len(rx_full) * 0.8)
SPLIT_SYMBOLS = int(len(tx_full) * 0.8)

# ==========================================
# 2. Unsupervised Dataset  (rx windows only, no labels)
# ==========================================
class PAM2DatasetCMA(Dataset):
    """
    Returns overlapping windows of raw received samples only.
    No labels are loaded or returned — CMA loss is self-referential.
    receptive_field and hop_samples match the original supervised setup
    so the exported weights slot directly into the same HLS datapath.
    """
    def __init__(self, rx_data, receptive_field=137, hop_samples=16):
        self.rx  = rx_data
        self.rf  = receptive_field
        self.hop = hop_samples

    def __len__(self):
        return (len(self.rx) - self.rf) // self.hop

    def __getitem__(self, idx):
        start = idx * self.hop
        x = self.rx[start : start + self.rf]
        return torch.tensor(x, dtype=torch.float32).unsqueeze(0)   # (1, 137)

train_loader = DataLoader(
    PAM2DatasetCMA(rx_full[:SPLIT_SAMPLES]),
    batch_size=64, shuffle=True
)
val_loader = DataLoader(
    PAM2DatasetCMA(rx_full[SPLIT_SAMPLES:]),
    batch_size=64, shuffle=False
)

# ==========================================
# 3. Network  — IDENTICAL to paper / HLS spec
#    Conv1: 1→5 ch, k=9, stride=8
#    Conv2: 5→5 ch, k=9, stride=1
#    Conv3: 5→8 ch, k=9, stride=2   ← Vp=8 parallel outputs
# ==========================================
class CNNEqualizerPaper(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv1d(1, 5, 9, stride=8, bias=True)
        self.bn1   = nn.BatchNorm1d(5)
        self.conv2 = nn.Conv1d(5, 5, 9, stride=1, bias=True)
        self.bn2   = nn.BatchNorm1d(5)
        self.conv3 = nn.Conv1d(5, 8, 9, stride=2, bias=False)  # Vp=8

    def forward(self, x):
        x = torch.relu(self.bn1(self.conv1(x)))
        x = torch.relu(self.bn2(self.conv2(x)))
        x = self.conv3(x)
        return x   # (B, 8, T_out)

model     = CNNEqualizerPaper()
optimizer = torch.optim.Adam(model.parameters(), lr=0.005)

# ==========================================
# 4. CMA / Godard Loss
# ==========================================
# PAM-2 alphabet {-1, +1}:  R² = E[a⁴]/E[a²] = 1/1 = 1.0
# Applied across ALL 8 output channels simultaneously.
# Each channel is independently driven toward the ±1 constellation.
R_SQUARED = 1.0

def cma_loss(y: torch.Tensor) -> torch.Tensor:
    """
    Godard Constant Modulus cost:  J = mean( (|y|² - R²)² )
    y shape: (B, 8, T_out) — loss averaged over batch, channels, and time.
    This keeps all 8 Vp output channels active with no label supervision.
    """
    return torch.mean((y.pow(2) - R_SQUARED).pow(2))

# ==========================================
# 5. Blind CMA Training Loop
# ==========================================
best_loss      = float('inf')
best_model_wts = copy.deepcopy(model.state_dict())

print("\nStarting Blind (CMA) Training — no labels used...")
print(f"Architecture: Conv1(1→5,k9,s8) → BN+ReLU → Conv2(5→5,k9,s1) → BN+ReLU → Conv3(5→8,k9,s2)")
print(f"CMA target R² = {R_SQUARED}  (PAM-2 ±1 alphabet)\n")

for epoch in range(150):
    # --- Train ---
    model.train()
    train_loss = 0.0
    for batch_x in train_loader:
        optimizer.zero_grad()
        out  = model(batch_x)          # (B, 8, T_out)
        loss = cma_loss(out)
        loss.backward()
        optimizer.step()
        train_loss += loss.item()
    train_loss /= len(train_loader)

    # --- Validate ---
    model.eval()
    val_loss = 0.0
    with torch.no_grad():
        for batch_x in val_loader:
            out       = model(batch_x)
            val_loss += cma_loss(out).item()
    val_loss /= len(val_loader)

    if val_loss < best_loss:
        best_loss      = val_loss
        best_model_wts = copy.deepcopy(model.state_dict())
        print(f"Epoch {epoch:03d} | train CMA: {train_loss:.6f} | val CMA: {val_loss:.6f}  ← best")

print(f"\nBlind Training Complete! Best Val CMA Loss: {best_loss:.6f}")

# ==========================================
# 6. Post-hoc BER  (labels used for measurement ONLY — not training)
# ==========================================
model.load_state_dict(best_model_wts)
model.eval()

class PAM2DatasetLabelled(Dataset):
    """Identical window logic to the original supervised dataset."""
    def __init__(self, rx_data, tx_labels, receptive_field=137, hop_samples=16, offset=50):
        self.rx     = rx_data
        self.tx     = tx_labels
        self.rf     = receptive_field
        self.hop    = hop_samples
        self.offset = offset

    def __len__(self):
        return min(
            (len(self.rx) - self.rf) // self.hop,
            (len(self.tx) - self.offset - 8) // (self.hop // 2)
        )

    def __getitem__(self, idx):
        start_rx = idx * self.hop
        x        = self.rx[start_rx : start_rx + self.rf]
        start_tx = (start_rx // 2) + self.offset
        y        = self.tx[start_tx : start_tx + 8]   # 8 symbols → Vp=8
        return (
            torch.tensor(x, dtype=torch.float32).unsqueeze(0),
            torch.tensor(y, dtype=torch.float32).unsqueeze(1)
        )

labelled_val = DataLoader(
    PAM2DatasetLabelled(rx_full[SPLIT_SAMPLES:], tx_full[SPLIT_SYMBOLS:]),
    batch_size=64, shuffle=False
)

errors = 0
total  = 0
with torch.no_grad():
    for batch_x, batch_y in labelled_val:
        out   = model(batch_x)
        preds = torch.where(out >= 0.0, 1.0, -1.0)
        errors += (preds != batch_y).sum().item()
        total  += batch_y.numel()

post_hoc_ber = errors / total if total > 0 else float('nan')
print(f"Post-hoc BER (labels used for measurement only): {post_hoc_ber:.4f}")

# ==========================================
# 7. Generate Golden Reference File for Vitis HLS
# ==========================================
print("\nGenerating golden_reference.dat for hardware verification...")
golden_preds = []
with torch.no_grad():
    for batch_x, _ in labelled_val:          # aligned symbol windows
        out   = model(batch_x)               # (B, 8, T_out)
        preds = torch.where(out >= 0.0, 1.0, -1.0)
        golden_preds.extend(preds.cpu().numpy().flatten().tolist())

os.makedirs('/mnt/user-data/outputs', exist_ok=True)
save_path = '/mnt/user-data/outputs/golden_reference.dat'
np.savetxt(save_path, golden_preds, fmt='%d')
print(f"SUCCESS! {len(golden_preds)} predictions written to: {save_path}")

# ==========================================
# 8. Fold BN into Conv and Export Weights for HLS
#    Matches ap_fixed<13,3> weight format
#    Vp=8, C=5, K=9
# ==========================================
def fold_conv_bn(conv, bn):
    scale    = bn.weight.detach() / torch.sqrt(bn.running_var + bn.eps)
    w_folded = conv.weight.detach() * scale.view(-1, 1, 1)
    b_folded = (conv.bias.detach() - bn.running_mean) * scale + bn.bias.detach()
    return w_folded, b_folded

w1, b1 = fold_conv_bn(model.conv1, model.bn1)
w2, b2 = fold_conv_bn(model.conv2, model.bn2)
w3     = model.conv3.weight.detach()   # (8, 5, 9)  — Vp=8, C=5, K=9

def export_layer_weights(tensor, name):
    flat    = tensor.numpy().flatten()
    arr_str = ", ".join([f"{val:.4f}" for val in flat])
    print(f"weight_t {name}[] = {{{arr_str}}};")

print("\n// --- COPY THESE WEIGHTS INTO YOUR HLS HEADER ---")
print(f"// Conv1 folded: shape (5, 1, 9)  — C=5, K=9")
export_layer_weights(w1, "l1_w")
export_layer_weights(b1, "l1_b")
print(f"// Conv2 folded: shape (5, 5, 9)  — C=5, C=5, K=9")
export_layer_weights(w2, "l2_w")
export_layer_weights(b2, "l2_b")
print(f"// Conv3:        shape (8, 5, 9)  — Vp=8, C=5, K=9")
export_layer_weights(w3, "l3_w")


Starting Blind (CMA) Training — no labels used...
Architecture: Conv1(1→5,k9,s8) → BN+ReLU → Conv2(5→5,k9,s1) → BN+ReLU → Conv3(5→8,k9,s2)
CMA target R² = 1.0  (PAM-2 ±1 alphabet)

Epoch 000 | train CMA: 0.621230 | val CMA: 0.760206  ← best
Epoch 001 | train CMA: 0.414586 | val CMA: 0.577885  ← best
Epoch 002 | train CMA: 0.264929 | val CMA: 0.239384  ← best
Epoch 003 | train CMA: 0.187417 | val CMA: 0.204720  ← best
Epoch 004 | train CMA: 0.132468 | val CMA: 0.186304  ← best
Epoch 005 | train CMA: 0.094741 | val CMA: 0.156625  ← best
Epoch 006 | train CMA: 0.069433 | val CMA: 0.126068  ← best
Epoch 007 | train CMA: 0.049816 | val CMA: 0.122906  ← best
Epoch 008 | train CMA: 0.037885 | val CMA: 0.103469  ← best
Epoch 009 | train CMA: 0.029276 | val CMA: 0.094736  ← best
Epoch 010 | train CMA: 0.024122 | val CMA: 0.078770  ← best
Epoch 012 | train CMA: 0.016229 | val CMA: 0.063190  ← best
Epoch 015 | train CMA: 0.011177 | val CMA: 0.054673  ← best
Epoch 017 | train CMA: 0.009779 | val 

In [16]:
import os
print("Your golden reference file is saved exactly here:")
print(os.path.abspath('golden_reference.dat'))

Your golden reference file is saved exactly here:
/content/golden_reference.dat
